# Day 10: Get World Cup 2026 Teams Data

This notebook fetches data for all 48 teams qualified for the 2026 FIFA World Cup.

**Data Sources:**
- football-data.org API: Current tournament stats
- WorldCupMatches.csv: Historical World Cup statistics

**Features for model:**
```python
FEATURES = [
    "fifa_rank",
    "group_stage_goals",
    "goals_conceded",
    "goal_differential",
    "wins_historical",
    "previous_world_cups",
    "win_percentage_historical"
]
```

In [12]:
import requests
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
API_KEY = os.getenv("FOOTBALL_API_KEY")

print("✅ Libraries loaded")
print(f"API Key found: {'Yes' if API_KEY else 'No'}")

✅ Libraries loaded
API Key found: Yes


## Step 1: Get Current 2026 Tournament Stats from football-data.org

In [13]:
# API configuration
headers = {"X-Auth-Token": API_KEY}
BASE_URL = "https://api.football-data.org/v4"

# Get World Cup standings (includes goals for/against per team)
url_standings = f"{BASE_URL}/competitions/WC/standings?season=2026"
print(f"Fetching: {url_standings}")

response = requests.get(url_standings, headers=headers)
print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    data_standings = response.json()
    print("✅ Standings data received")
else:
    print(f"❌ Error: {response.text}")

Fetching: https://api.football-data.org/v4/competitions/WC/standings?season=2026
Status Code: 200
✅ Standings data received


In [14]:
# Extract team stats from standings
teams_2026 = []

for standing in data_standings.get("standings", []):
    if standing.get("type") == "TOTAL":
        for table in standing.get("table", []):
            team = table.get("team", {})
            teams_2026.append({
                "team_id": team.get("id"),
                "country_name": team.get("name"),
                "fifa_code": team.get("tla"),
                "group_stage_goals": table.get("goalsFor", 0),
                "goals_conceded": table.get("goalsAgainst", 0),
                "goal_differential": table.get("goalDifference", 0),
                "wins_current": table.get("won", 0),
                "draws_current": table.get("draw", 0),
                "losses_current": table.get("lost", 0),
                "points": table.get("points", 0)
            })

df_current = pd.DataFrame(teams_2026)
print(f"✅ {len(df_current)} teams extracted from standings")
df_current.head(10)

✅ 48 teams extracted from standings


,team_id,country_name,fifa_code,group_stage_goals,goals_conceded,goal_differential,wins_current,draws_current,losses_current,points
0,773,France,FRA,10,2,8,3,0,0,9
1,762,Argentina,ARG,8,1,7,3,0,0,9
2,769,Mexico,MEX,6,0,6,3,0,0,9
3,8601,Netherlands,NED,10,4,6,2,1,0,7
4,764,Brazil,BRA,7,1,6,2,1,0,7
5,760,Spain,ESP,5,0,5,2,1,0,7
6,788,Switzerland,SUI,7,3,4,2,1,0,7
7,770,England,ENG,6,2,4,2,1,0,7
8,815,Morocco,MAR,6,3,3,2,1,0,7
9,818,Colombia,COL,4,1,3,2,1,0,7


## Step 2: Calculate Historical Stats from WorldCupMatches.csv

In [15]:
# Load historical World Cup matches data
df_matches = pd.read_csv("../data/raw/WorldCupMatches.csv")
print(f"✅ Historical data loaded: {len(df_matches)} matches")

✅ Historical data loaded: 4572 matches


In [16]:
# Clean team names - remove HTML artifacts like 'rn">'
def clean_team_name(name):
    if pd.isna(name):
        return name
    if isinstance(name, str) and 'rn">' in name:
        name = name.split('rn">')[-1]
    return name.strip()

df_matches["Home Team Name"] = df_matches["Home Team Name"].apply(clean_team_name)
df_matches["Away Team Name"] = df_matches["Away Team Name"].apply(clean_team_name)

# Get all unique team names from historical data
home_teams = set(df_matches["Home Team Name"].dropna().unique())
away_teams = set(df_matches["Away Team Name"].dropna().unique())
all_historical_teams = home_teams | away_teams

print(f"Total unique teams in history: {len(all_historical_teams)}")
print(f"\nAll teams:")
for team in sorted(all_historical_teams):
    print(f"  - {team}")

Total unique teams in history: 83

All teams:
  - Algeria
  - Angola
  - Argentina
  - Australia
  - Austria
  - Belgium
  - Bolivia
  - Bosnia and Herzegovina
  - Brazil
  - Bulgaria
  - Cameroon
  - Canada
  - Chile
  - China PR
  - Colombia
  - Costa Rica
  - Croatia
  - Cuba
  - Czech Republic
  - Czechoslovakia
  - C�te d'Ivoire
  - Denmark
  - Dutch East Indies
  - Ecuador
  - Egypt
  - El Salvador
  - England
  - France
  - German DR
  - Germany
  - Germany FR
  - Ghana
  - Greece
  - Haiti
  - Honduras
  - Hungary
  - IR Iran
  - Iran
  - Iraq
  - Israel
  - Italy
  - Jamaica
  - Japan
  - Korea DPR
  - Korea Republic
  - Kuwait
  - Mexico
  - Morocco
  - Netherlands
  - New Zealand
  - Nigeria
  - Northern Ireland
  - Norway
  - Paraguay
  - Peru
  - Poland
  - Portugal
  - Republic of Ireland
  - Romania
  - Russia
  - Saudi Arabia
  - Scotland
  - Senegal
  - Serbia
  - Serbia and Montenegro
  - Slovakia
  - Slovenia
  - South Africa
  - Soviet Union
  - Spain
  - Sweden
  -

In [17]:
# Mapping: historical name -> name used in 2026 API data
NAME_MAPPING = {
    # Teams that kept the same name
    "Brazil": "Brazil",
    "Argentina": "Argentina",
    "Uruguay": "Uruguay",
    "Italy": "Italy",
    "Germany": "Germany",
    "France": "France",
    "England": "England",
    "Spain": "Spain",
    "Netherlands": "Netherlands",
    "Belgium": "Belgium",
    "Croatia": "Croatia",
    "Morocco": "Morocco",
    "Japan": "Japan",
    "Korea Republic": "Korea Republic",
    "Portugal": "Portugal",
    "Mexico": "Mexico",
    "USA": "USA",
    "Colombia": "Colombia",
    "Senegal": "Senegal",
    "Ecuador": "Ecuador",
    "Australia": "Australia",
    "Switzerland": "Switzerland",
    "Cameroon": "Cameroon",
    "Ghana": "Ghana",
    "Tunisia": "Tunisia",
    "Saudi Arabia": "Saudi Arabia",
    "IR Iran": "IR Iran",
    "Iran": "IR Iran",
    "Qatar": "Qatar",
    "Canada": "Canada",
    "New Zealand": "New Zealand",
    "Algeria": "Algeria",
    "Egypt": "Egypt",
    "Paraguay": "Paraguay",
    "Chile": "Chile",
    "Peru": "Peru",
    "Nigeria": "Nigeria",
    "Serbia": "Serbia",
    "Poland": "Poland",
    "Denmark": "Denmark",
    "Sweden": "Sweden",
    "Norway": "Norway",
    "Austria": "Austria",
    "Czech Republic": "Czechia",
    "Scotland": "Scotland",
    "Wales": "Wales",
    "Ukraine": "Ukraine",
    "Romania": "Romania",
    "Hungary": "Hungary",
    "Slovakia": "Slovakia",
    "Slovenia": "Slovenia",
    "South Africa": "South Africa",
    "Iraq": "Iraq",
    "Haiti": "Haiti",
    "Jamaica": "Jamaica",
    "Trinidad and Tobago": "Trinidad and Tobago",
    "United Arab Emirates": "United Arab Emirates",
    "Kuwait": "Kuwait",
    "Greece": "Greece",
    "Israel": "Israel",
    "Honduras": "Honduras",
    "El Salvador": "El Salvador",
    "Bulgaria": "Bulgaria",
    "Costa Rica": "Costa Rica",
    "Bolivia": "Bolivia",
    "Togo": "Togo",
    "Angola": "Angola",
    "China PR": "China PR",
    "Northern Ireland": "Northern Ireland",
    "Republic of Ireland": "Republic of Ireland",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    # Historical names -> Modern equivalents
    "Germany FR": "Germany",
    "German DR": "Germany",
    "Czechoslovakia": "Czechia",
    "Soviet Union": "Russia",
    "Yugoslavia": "Serbia",
    "Serbia and Montenegro": "Serbia",
    "Zaire": "Congo DR",
    "Dutch East Indies": "Indonesia",
    "Korea DPR": "Korea DPR",
    "Côte d'Ivoire": "Ivory Coast",
}

print(f"✅ Name mapping created with {len(NAME_MAPPING)} entries")

✅ Name mapping created with 81 entries


In [18]:
# Calculate historical statistics for each team
historical_stats = []

for team_name in all_historical_teams:
    # As home team
    home_matches = df_matches[df_matches["Home Team Name"] == team_name]
    home_wins = (home_matches["Home Team Goals"] > home_matches["Away Team Goals"]).sum()
    
    # As away team
    away_matches = df_matches[df_matches["Away Team Name"] == team_name]
    away_wins = (away_matches["Away Team Goals"] > away_matches["Home Team Goals"]).sum()
    
    total_wins = home_wins + away_wins
    total_matches = len(home_matches) + len(away_matches)
    
    # Count unique World Cup years participated
    home_years = set(home_matches["Year"].dropna().unique())
    away_years = set(away_matches["Year"].dropna().unique())
    all_years = home_years | away_years
    
    historical_stats.append({
        "country_name": team_name,
        "wins_historical": total_wins,
        "previous_world_cups": len(all_years),
        "win_percentage_historical": round(100 * total_wins / total_matches, 1) if total_matches > 0 else 0
    })

df_historical = pd.DataFrame(historical_stats)

# Map to 2026 API names
df_historical["mapped_name"] = df_historical["country_name"].map(NAME_MAPPING)

# Check for unmapped teams
unmapped = df_historical[df_historical["mapped_name"].isna()]
if len(unmapped) > 0:
    print("⚠️ Unmapped teams (excluded from merge):")
    for team in unmapped["country_name"].tolist():
        print(f"  - {team}")
else:
    print("✅ All teams mapped successfully")

# Filter only mapped teams
df_historical = df_historical.dropna(subset=["mapped_name"])

# Aggregate historical stats for teams that map to the same modern name
df_historical_agg = df_historical.groupby("mapped_name").agg({
    "wins_historical": "sum",
    "previous_world_cups": "sum",
    "win_percentage_historical": "mean"
}).reset_index()
df_historical_agg.columns = ["country_name", "wins_historical", "previous_world_cups", "win_percentage_historical"]

print(f"\n✅ Historical stats aggregated for {len(df_historical_agg)} modern teams")
df_historical_agg.sort_values("wins_historical", ascending=False).head(10)

⚠️ Unmapped teams (excluded from merge):
  - C�te d'Ivoire
  - Turkey
  - Cuba
  - Russia

✅ Historical stats aggregated for 73 modern teams


,country_name,wins_historical,previous_world_cups,win_percentage_historical
25,Germany,72,19,54.066667
8,Brazil,71,20,65.700000
35,Italy,45,18,54.200000
2,Argentina,44,16,54.300000
43,Netherlands,29,10,53.700000
24,France,29,14,47.500000
62,Spain,29,14,49.200000
23,England,26,14,41.900000
71,Uruguay,20,12,38.500000
58,Serbia,17,11,25.500000


## Step 3: Merge All Data and Create Final CSV

In [19]:
# Merge current 2026 data with historical stats
df_final = df_current.merge(
    df_historical_agg,
    on="country_name",
    how="left"
)

# Fill missing historical data with 0
df_final["wins_historical"] = df_final["wins_historical"].fillna(0).astype(int)
df_final["previous_world_cups"] = df_final["previous_world_cups"].fillna(0).astype(int)
df_final["win_percentage_historical"] = df_final["win_percentage_historical"].fillna(0)

# Add placeholder for fifa_rank (to be filled manually or from another source)
df_final["fifa_rank"] = range(1, len(df_final) + 1)

print(f"✅ Final dataset created: {len(df_final)} teams")
print(f"\nTeams with historical data: {df_final[df_final['wins_historical'] > 0].shape[0]}")
print(f"Teams without historical data: {df_final[df_final['wins_historical'] == 0].shape[0]}")
df_final.head(10)

✅ Final dataset created: 48 teams

Teams with historical data: 30
Teams without historical data: 18


,team_id,country_name,fifa_code,group_stage_goals,goals_conceded,goal_differential,wins_current,draws_current,losses_current,points,wins_historical,previous_world_cups,win_percentage_historical,fifa_rank
0,773,France,FRA,10,2,8,3,0,0,9,29,14,47.5,1
1,762,Argentina,ARG,8,1,7,3,0,0,9,44,16,54.3,2
2,769,Mexico,MEX,6,0,6,3,0,0,9,14,15,25.9,3
3,8601,Netherlands,NED,10,4,6,2,1,0,7,29,10,53.7,4
4,764,Brazil,BRA,7,1,6,2,1,0,7,71,20,65.7,5
5,760,Spain,ESP,5,0,5,2,1,0,7,29,14,49.2,6
6,788,Switzerland,SUI,7,3,4,2,1,0,7,11,10,32.4,7
7,770,England,ENG,6,2,4,2,1,0,7,26,14,41.9,8
8,815,Morocco,MAR,6,3,3,2,1,0,7,2,4,15.4,9
9,818,Colombia,COL,4,1,3,2,1,0,7,8,5,40.0,10


## Step 4: Select Final Columns and Save

In [20]:
# Select and rename columns to match required FEATURES
df_output = df_final[[
    "country_name",
    "fifa_code",
    "fifa_rank",
    "group_stage_goals",
    "goals_conceded",
    "goal_differential",
    "wins_historical",
    "previous_world_cups",
    "win_percentage_historical"
]].copy()

# Sort by goal differential (best teams first)
df_output = df_output.sort_values("goal_differential", ascending=False)

print("\n" + "=" * 60)
print("WORLD CUP 2026 - ALL 48 TEAMS")
print("=" * 60)
print(df_output.to_string(index=False))


WORLD CUP 2026 - ALL 48 TEAMS
      country_name fifa_code  fifa_rank  group_stage_goals  goals_conceded  goal_differential  wins_historical  previous_world_cups  win_percentage_historical
            France       FRA          1                 10               2                  8               29                   14                  47.500000
         Argentina       ARG          2                  8               1                  7               44                   16                  54.300000
            Mexico       MEX          3                  6               0                  6               14                   15                  25.900000
       Netherlands       NED          4                 10               4                  6               29                   10                  53.700000
            Brazil       BRA          5                  7               1                  6               71                   20                  65.700000
           Germ

In [21]:
# Save to CSV
output_path = "../data/raw/rankings_2026.csv"
df_output.to_csv(output_path, index=False)

print(f"\n✅ Data saved to: {output_path}")
print(f"   Total teams: {len(df_output)}")
print(f"   Columns: {df_output.columns.tolist()}")


✅ Data saved to: ../data/raw/rankings_2026.csv
   Total teams: 48
   Columns: ['country_name', 'fifa_code', 'fifa_rank', 'group_stage_goals', 'goals_conceded', 'goal_differential', 'wins_historical', 'previous_world_cups', 'win_percentage_historical']


## Step 5: Verify Features for Model

In [22]:
# Verify all required features are present
REQUIRED_FEATURES = [
    "fifa_rank",
    "group_stage_goals",
    "goals_conceded",
    "goal_differential",
    "wins_historical",
    "previous_world_cups",
    "win_percentage_historical"
]

print("Feature verification:")
print("-" * 50)

all_features_present = True
for feature in REQUIRED_FEATURES:
    if feature in df_output.columns:
        print(f"✅ {feature}: min={df_output[feature].min()}, max={df_output[feature].max()}, mean={df_output[feature].mean():.1f}")
    else:
        print(f"❌ {feature}: MISSING")
        all_features_present = False

print("-" * 50)
if all_features_present:
    print("\n✅ ALL REQUIRED FEATURES ARE PRESENT")
else:
    print("\n❌ SOME FEATURES ARE MISSING")

Feature verification:
--------------------------------------------------
✅ fifa_rank: min=1, max=48, mean=24.5
✅ group_stage_goals: min=0, max=10, mean=4.5
✅ goals_conceded: min=0, max=12, mean=4.5
✅ goal_differential: min=-11, max=8, mean=0.0
✅ wins_historical: min=0, max=72, mean=9.7
✅ previous_world_cups: min=0, max=20, mean=5.4
✅ win_percentage_historical: min=0.0, max=65.7, mean=21.8
--------------------------------------------------

✅ ALL REQUIRED FEATURES ARE PRESENT


In [23]:
# Show top 10 teams by goal differential
print("\nTOP 10 TEAMS BY GOAL DIFFERENTIAL:")
print("=" * 70)
top10 = df_output.nlargest(10, "goal_differential")
print(top10.to_string(index=False))


TOP 10 TEAMS BY GOAL DIFFERENTIAL:
 country_name fifa_code  fifa_rank  group_stage_goals  goals_conceded  goal_differential  wins_historical  previous_world_cups  win_percentage_historical
       France       FRA          1                 10               2                  8               29                   14                  47.500000
    Argentina       ARG          2                  8               1                  7               44                   16                  54.300000
       Mexico       MEX          3                  6               0                  6               14                   15                  25.900000
  Netherlands       NED          4                 10               4                  6               29                   10                  53.700000
       Brazil       BRA          5                  7               1                  6               71                   20                  65.700000
      Germany       GER         11      

## Notes

1. **FIFA Ranking**: The `fifa_rank` column is currently sequential. Update with actual FIFA rankings from:
   - https://www.fifa.com/fifa-world-ranking/men

2. **Historical Data**: Stats calculated from all World Cup matches (1930-2022).
   - Historical teams (Germany FR, Czechoslovakia, etc.) are mapped to modern equivalents.

3. **Next Steps**: Use this CSV in the prediction notebook to generate 2026 World Cup predictions.